In [16]:
import pandas as pd
import numpy as np

np.set_printoptions(threshold=np.inf)

In [17]:
# Loading guest data from the restaurant
guest_df = pd.read_csv("../data/raw/WEEVA_GUESTS.csv")
print(guest_df.head(1))

# Rename the column
guest_df.rename(columns={"DATE" : 'Date'}, inplace=True)

# Convert to datetime
guest_df["Date"] = pd.to_datetime(guest_df["Date"])


print(guest_df.head(1))

        DATE GUESTS
0  11/1/2018      0
        Date GUESTS
0 2018-11-01      0


In [18]:
# Loading weather data by time frame
weather_nov_2018_may_2021_df       = pd.read_csv("../data/raw/Groningen 2018-11-01 to 2021-05-31.csv")
weather_june_2021_december_2023_df = pd.read_csv("../data/raw/Groningen 2021-06-01 to 2023-12-31.csv")
weather_jan_2024_april_2025_df     = pd.read_csv("../data/raw/Groningen 2024-01-01 to 2025-04-28.csv")

# Combining the weather datasets into one dataframe
weather_df = pd.concat([weather_nov_2018_may_2021_df, 
                        weather_june_2021_december_2023_df, 
                        weather_jan_2024_april_2025_df], 
                        ignore_index=True)

# Rename the column
weather_df.rename(columns={"datetime" : 'Date'}, inplace=True)

# Convert to datetime
weather_df["Date"] = pd.to_datetime(weather_df["Date"])

irrelevant_features = ['name', 'dew', 'precipcover', 'snow', 'snowdepth',
                        'winddir', 'sealevelpressure', 'visibility', 
                        'solarenergy', 'severerisk', 'sunrise', 'sunset',
                        'moonphase', 'description', 'stations', 'icon', 
                        'conditions']
weather_df = weather_df.drop(columns=irrelevant_features)

# Create boolean columns to encode preciptype
weather_df['rain'] = weather_df['preciptype'].str.contains('rain', na=False)\
                                                                .astype(int)
weather_df['snow'] = weather_df['preciptype'].str.contains('snow', na=False)\
                                                                .astype(int)

# Drop preciptype column
weather_df = weather_df.drop(columns='preciptype')

print(weather_df.columns)
print(weather_df.head())
print(weather_df.shape)

Index(['Date', 'tempmax', 'tempmin', 'temp', 'feelslikemax', 'feelslikemin',
       'feelslike', 'humidity', 'precip', 'precipprob', 'windgust',
       'windspeed', 'cloudcover', 'solarradiation', 'uvindex', 'rain', 'snow'],
      dtype='object')
        Date  tempmax  tempmin  temp  feelslikemax  feelslikemin  feelslike  \
0 2018-11-01     12.9      7.2   9.5          12.9           4.6        8.0   
1 2018-11-02     11.0      1.7   8.2          11.0          -1.1        6.7   
2 2018-11-03     10.0      0.6   4.6          10.0          -1.9        2.7   
3 2018-11-04     10.7     -0.4   5.2          10.7          -2.5        3.4   
4 2018-11-05     10.1      5.9   8.8          10.1           4.3        7.9   

   humidity  precip  precipprob  windgust  windspeed  cloudcover  \
0      83.5   2.532         100      35.0       16.3        21.5   
1      85.6   1.095         100      49.4       22.5        40.6   
2      89.2   0.000           0      30.9       15.5         6.1   
3     

In [19]:
# Loading data on holidays and calendar dates
school_holidays_df           = pd.read_csv("../data/raw/groningen_school_holidays_boolean.csv")
public_holidays_groningen_df = pd.read_csv("../data/raw/public_holidays_2018_2025.csv")
public_holidays_germany_df   = pd.read_csv("../data/raw/public_holidays_germany_2018_2025.csv")
calendar_df                  = pd.read_csv("../data/raw/dates_with_weekdays.csv")

In [20]:
# Convert calendar_df["Date"] to pd.to_datetime
calendar_df["Date"] = pd.to_datetime(calendar_df["Date"])

# Encode Day of the week into separate columns
dow_dummies = pd.get_dummies(calendar_df['DayOfWeek'], prefix='is', dtype=int)
calendar_df = pd.concat([calendar_df, dow_dummies], axis=1)

# Drop DayOfWeek and IsWeekend
calendar_df = calendar_df.drop(columns=['DayOfWeek', 'IsWeekend']) 

print(calendar_df.head(1))

        Date  is_Friday  is_Monday  is_Saturday  is_Sunday  is_Thursday  \
0 2018-11-01          0          0            0          0            1   

   is_Tuesday  is_Wednesday  
0           0             0  


In [21]:
school_holidays_df.tail()
school_holidays_df.shape
# school_holidays_df.dtypes

(3035, 2)

In [22]:
school_holidays_df["Date"] = pd.to_datetime(school_holidays_df["Date"])

start_date = '2018-11-01'
end_date = '2025-04-28'

# Filter to keep only dates within the desired range
school_holidays_df = school_holidays_df[
    (school_holidays_df['Date'] >= start_date) &
    (school_holidays_df['Date'] <= end_date)
]

# Remove duplicate dates, keeping the last occurrence
school_holidays_df = school_holidays_df.drop_duplicates(subset='Date', keep='last')

school_holidays_bool_df = pd.DataFrame(school_holidays_df)
# Convert 'Yes'/'No' to True/False in a specific column (e.g., 'IsHoliday')
school_holidays_bool_df['IsHoliday'] = school_holidays_bool_df['IsHoliday'].map({"Yes": 1, "No": 0})

school_holidays_bool_df.shape

C:\Users\Matei\AppData\Local\Temp\ipykernel_32524\4025799754.py:1: UserWarning: Parsing dates in %d.%m.%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  school_holidays_df["Date"] = pd.to_datetime(school_holidays_df["Date"])


(2371, 2)

In [23]:
# 9 entries out of our date range for groningen
# public_holidays_groningen_df.head(50)
# public_holidays_groningen_df.tail(20)

public_holidays_groningen_df["Holiday"].unique()
# Only 9 unique holidays, but inconsistant naming () 
# e.g: 'Koningsdag (National Holiday)' / 'Koningsdag (National Day)'
# public_holidays_groningen_df["Holiday"].unique().shape

public_holidays_groningen_df["Holiday"] = public_holidays_groningen_df\
                                                       ['Holiday'].str.strip()

# public_holidays_groningen_df["Holiday"].unique()

name_map = {
    "New Year": "New Year's Day",
    "Koningsdag (National Holiday)": "King\'s Day",
     "Koningsdag (National Day)": "King\'s Day",
     "St. Stephen's Day": "Second Christmas Day"
}

# Solve inconsistant naming
public_holidays_groningen_df["Holiday"] = public_holidays_groningen_df\
                                                 ['Holiday'].replace(name_map)

public_holidays_groningen_df["Holiday"].unique()
# # Only 7 unique holidays after filtering
# public_holidays_groningen_df["Holiday"].unique().shape

# public_holidays_groningen_df.head()


array(["New Year's Day", 'Easter Monday', "King's Day", 'Ascension Day',
       'Whit Monday', 'Christmas', 'Second Christmas Day'], dtype=object)

In [24]:
public_holidays_germany_df["Holiday"].unique()

public_holidays_germany_df["Holiday"] = public_holidays_germany_df\
                                                       ['Holiday'].str.strip()

name_map = {
    "New Years Day": "New Year's Day",
    "Christmas Day": "Christmas",
    "Boxing Day": "Second Christmas Day"
}

# Solve inconsistant naming
public_holidays_germany_df["Holiday"] = public_holidays_germany_df\
                                                 ['Holiday'].replace(name_map)

public_holidays_germany_df["Holiday"].unique()


array(["New Year's Day", 'Good Friday', 'Easter Monday', 'May Day',
       'Ascension Day', 'Whit Monday', 'Day of German Unity', 'Christmas',
       'Second Christmas Day'], dtype=object)

In [25]:
# Setting the date column to the right data type
public_holidays_groningen_df["Date"] = pd.to_datetime(
                                        public_holidays_groningen_df["Date"],
                                        format="%d.%m.%Y")
public_holidays_germany_df["Date"] = pd.to_datetime(
                                        public_holidays_germany_df["Date"],
                                        format="%d.%m.%Y")

# Combine all the holidays
combined_holidays_df = pd.concat([public_holidays_groningen_df,
                                   public_holidays_germany_df],
                                    ignore_index=True)

combined_holidays_df["Holiday"].unique().shape

combined_holidays_df['is_holiday'] = 1


# Drop duplicates
combined_holidays_df = combined_holidays_df.pivot_table(
    index='Date',
    columns='Holiday',
    values='is_holiday',
    fill_value=0
).reset_index()

combined_holidays_df.shape

# Defining a range of dates for the full holiday dataframe
date_range = pd.date_range(start='2018-11-01', end='2025-04-28', freq='D')

# Create a new DataFrame with that full date range
full_date_range_df = pd.DataFrame({'Date': date_range})

# Merge on Date — left join to preserve full date range
merged_df = full_date_range_df.merge(combined_holidays_df,
                                     on='Date',
                                     how='left')

# # Fill NaN's 
final_holiday_df = merged_df.fillna('0').astype({col: 'int' for col in \
                                                     merged_df.columns \
                                                        if col != 'Date'})

final_holiday_df.head()
final_holiday_df.columns




Index(['Date', 'Ascension Day', 'Christmas', 'Day of German Unity',
       'Easter Monday', 'Good Friday', 'King's Day', 'May Day',
       'New Year's Day', 'Second Christmas Day', 'Whit Monday'],
      dtype='object')

In [26]:
print(calendar_df.head(1))
print(calendar_df.tail(1))


        Date  is_Friday  is_Monday  is_Saturday  is_Sunday  is_Thursday  \
0 2018-11-01          0          0            0          0            1   

   is_Tuesday  is_Wednesday  
0           0             0  
           Date  is_Friday  is_Monday  is_Saturday  is_Sunday  is_Thursday  \
2370 2025-04-28          0          1            0          0            0   

      is_Tuesday  is_Wednesday  
2370           0             0  


In [29]:
# Loading data on number of items ordered in the restaurant
course_data_2018_2022_df = pd.read_csv("../data/raw/Weeva_data_2018-2022.csv")
course_data_2023_2025_df = pd.read_csv("../data/raw/Weeva_Data_2023-x.csv")

course_data_df = pd.concat([course_data_2018_2022_df, 
                            course_data_2023_2025_df], 
                            ignore_index=True)

# Setting the date column to the right data type
course_data_df["Date"] = pd.to_datetime(course_data_df["Date"],
                                        format="%d-%m-%Y")

article_map = {
    "broodplankje": "art_broodplankje",
    "captain.*dinner": "art_captain_dinner",
    "(?<!\w\s)cr.me.*br.l.e(?!\s)": "art_creme_brulee",
    "dame.*blanche(?!\s)": "art_dame_blanche",
    "sliptong.*meuni.re": "art_sliptong",
    "garnalen.*cocktail": "art_garnalen_cocktail",
    "bloedworst": "art_bloedworst",
    "olijven": "art_olijven",
    "kaasplankje(?!\s)": "art_kaasplankje",
    "(?<!\w\s)kalfslever": "art_kalfslever",
    "koffie.*compleet(?!\s)": "art_koffie_compleet",
    "groningse poffert": "art_poffert",
    "runder.*carpaccio": "art_carpaccio",
    "sat.*spies(?!\s)": "art_sate_spies",
    "schnitzel": "art_schnitzel",
    "sorbet.*weeva": "art_sorbet",
    "andijvie stamppot": "art_stamppot",
    "vers.*markt": "art_vers_van_de_markt",
    "weeva.*gehaktbal(?!\s)": "art_gehaktbal",
    "weeva.*spareribs(?!\s)": "art_spareribs",
    "tournedos(?!\s)": "art_tournedos",
    "zalmfilet(?!\s)": "art_zalmfilet",
}

# Lower case
course_data_df["Article"] = course_data_df["Article"].str.lower()

# Solve inconsistant naming
course_data_df["Article"] = course_data_df["Article"].replace(article_map,\
                                                               regex=True)

# Keep only rows where the Article value is one of the mapped values
valid_articles = set(article_map.values())
course_data_df = course_data_df[course_data_df["Article"].\
                                                        isin(valid_articles)]

# print(course_data_df["Article"].unique())
# print(course_data_df.head())

course_data_df = course_data_df.pivot_table(
    index="Date",
    columns="Article",
    values="Sold articles amount",
    fill_value=0,
    aggfunc="sum",
    ).reset_index()

course_data_df.head()


Article,Date,art_bloedworst,art_broodplankje,art_captain_dinner,art_carpaccio,art_creme_brulee,art_dame_blanche,art_garnalen_cocktail,art_gehaktbal,art_kaasplankje,...,art_poffert,art_sate_spies,art_schnitzel,art_sliptong,art_sorbet,art_spareribs,art_stamppot,art_tournedos,art_vers_van_de_markt,art_zalmfilet
0,2018-11-27,0,3,0,11,0,5,2,0,0,...,5,6,8,1,7,6,0,0,0,2
1,2018-11-28,0,7,0,3,7,3,0,2,2,...,3,5,3,7,2,3,0,0,0,5
2,2018-11-29,4,9,0,7,10,11,6,10,1,...,3,6,8,6,1,6,0,0,0,6
3,2018-11-30,3,12,0,5,5,14,2,0,0,...,6,11,10,2,3,13,0,0,0,3
4,2018-12-01,7,13,0,10,5,15,7,10,2,...,8,13,10,15,5,9,0,0,0,8


In [30]:
# ------------------------------------------------------------------
# 0. counts_only : Date‑indexed DataFrame of raw counts per article
# ------------------------------------------------------------------
counts_only = course_data_df.set_index('Date')
article_names = counts_only.columns.tolist()
A = len(article_names)

# ------------------------------------------------------------------
# ORDINAL‑rank table  (1 = top seller)
# ------------------------------------------------------------------
rank_int_df = (
    counts_only
      .rank(axis=1, method='first', ascending=False)   # break ties by col order
      .astype(int)
      .reset_index()
)

# ------------------------------------------------------------------
# Order-as‑list column  (['art_schnitzel', 'art_broodplankje', ...])
# ------------------------------------------------------------------
def order_list(row):
    # row is a Series of counts   -> return names sorted desc
    return list(row.sort_values(ascending=False).index)

rank_order_df = pd.DataFrame({
    'Date': counts_only.index,
    'rank_order': counts_only.apply(order_list, axis=1)
})

# ------------------------------------------------------------------
# Index‑vector column ([0, 4, 2, ...]  same length A)
# ------------------------------------------------------------------
article_to_idx = {name: i for i, name in enumerate(article_names)}

rank_idx_vec_df = rank_order_df.copy()
rank_idx_vec_df['rank_idx_vec'] = rank_idx_vec_df['rank_order']\
                                      .apply(lambda lst: [article_to_idx[a] for a in lst])
rank_idx_vec_df = rank_idx_vec_df.drop(columns='rank_order')


# helper: ordinal rank -> one‑hot list
def to_onehot(rank, length=A):
    vec = [0] * length
    # rank is 1‑based, so index = rank‑1
    vec[rank - 1] = 1
    return vec

# ------------------------------------------
# Build the nested one‑hot DataFrame
# ------------------------------------------
rank_1hot_nested_df = rank_int_df.copy()      # Date + ordinal ranks

for col in article_names:
    rank_1hot_nested_df[col] = rank_1hot_nested_df[col].apply(to_onehot)


In [31]:
print(course_data_df.head(1)["Date"])

print(course_data_df.tail(1)["Date"])

print(f"{course_data_df.shape=}")


print("First date in final_holiday_df =", final_holiday_df["Date"].head(1))
print("Last date in final_holiday_df  =", final_holiday_df["Date"].tail(1))

print(f"{final_holiday_df.shape=}")

print(final_holiday_df["Date"].unique())


course_dates = set(course_data_df["Date"])
holiday_dates = set(final_holiday_df["Date"])

# Dates in course_data_df but not in final_holiday_df
only_in_course = course_dates - holiday_dates
print("Dates in course_data_df but NOT in final_holiday_df:")
print(sorted(only_in_course))

# Dates in final_holiday_df but not in course_data_df
only_in_holiday = holiday_dates - course_dates
print("\nDates in final_holiday_df but NOT in course_data_df:")
print(sorted(only_in_holiday))

print(f"Number of missing datapoints = {len(only_in_holiday)}")


0   2018-11-27
Name: Date, dtype: datetime64[ns]
2244   2025-04-28
Name: Date, dtype: datetime64[ns]
course_data_df.shape=(2245, 23)
First date in final_holiday_df = 0   2018-11-01
Name: Date, dtype: datetime64[ns]
Last date in final_holiday_df  = 2370   2025-04-28
Name: Date, dtype: datetime64[ns]
final_holiday_df.shape=(2371, 11)
<DatetimeArray>
['2018-11-01 00:00:00', '2018-11-02 00:00:00', '2018-11-03 00:00:00',
 '2018-11-04 00:00:00', '2018-11-05 00:00:00', '2018-11-06 00:00:00',
 '2018-11-07 00:00:00', '2018-11-08 00:00:00', '2018-11-09 00:00:00',
 '2018-11-10 00:00:00',
 ...
 '2025-04-19 00:00:00', '2025-04-20 00:00:00', '2025-04-21 00:00:00',
 '2025-04-22 00:00:00', '2025-04-23 00:00:00', '2025-04-24 00:00:00',
 '2025-04-25 00:00:00', '2025-04-26 00:00:00', '2025-04-27 00:00:00',
 '2025-04-28 00:00:00']
Length: 2371, dtype: datetime64[ns]
Dates in course_data_df but NOT in final_holiday_df:
[]

Dates in final_holiday_df but NOT in course_data_df:
[Timestamp('2018-11-01 00:00:00

In [ ]:
import os
import pickle as pkl

# ----------------------------------------
# OUTPUT DIR
# ----------------------------------------
output_dir = "../data/cleaned/"
os.makedirs(output_dir, exist_ok=True)

# ----------------------------------------
# COVID MASKING
# ----------------------------------------
COVID_WINDOWS = [
    ("2020-03-01", "2020-05-31"),
    ("2020-12-01", "2021-06-30"),
    ("2021-11-01", "2022-01-31"),
]

def in_covid_period(s, windows=COVID_WINDOWS):
    mask = pd.Series(False, index=s.index)
    for start, end in windows:
        mask |= s.between(start, end)
    return mask

# ----------------------------------------
# Align by shared dates
# ----------------------------------------
# Base features
base_dfs = [
    guest_df,
    calendar_df,
    weather_df,
    school_holidays_bool_df,
    final_holiday_df,
    rank_int_df,
    rank_1hot_nested_df,
]

# Compute common dates
common_idx = pd.DatetimeIndex(base_dfs[0]["Date"].unique())
for df in base_dfs[1:]:
    common_idx = common_idx.intersection(pd.DatetimeIndex(df["Date"].unique()))

# Filter all to shared dates
for i in range(len(base_dfs)):
    base_dfs[i] = base_dfs[i][base_dfs[i]["Date"].isin(common_idx)].reset_index(drop=True)

# Unpack back
(guest_df,
 calendar_df,
 weather_df,
 school_holidays_bool_df,
 final_holiday_df,
 rank_int_df,
 rank_1hot_nested_df) = base_dfs

# ----------------------------------------
# Remove guest outliers + COVID dates
# ----------------------------------------
guest_df['GUESTS'] = pd.to_numeric(guest_df['GUESTS'], errors='coerce')
guest_df = guest_df[
    (guest_df['GUESTS'].between(1, 200)) &
    (~in_covid_period(guest_df['Date']))
].reset_index(drop=True)

valid_dates = set(guest_df['Date'])

# Filter all others based on valid guest dates and COVID
filtered = {}
for name, df in {
    'calendar': calendar_df,
    'weather': weather_df,
    'school_holidays': school_holidays_bool_df,
    'public_holidays': final_holiday_df,
    'rank_int': rank_int_df,
    'rank_1hot': rank_1hot_nested_df,
}.items():
    df = df[
        df['Date'].isin(valid_dates) &
        (~in_covid_period(df["Date"]))
    ].reset_index(drop=True)
    filtered[name] = df

# ----------------------------------------
# Save base components (optional)
# ----------------------------------------
# Save base filtered CSVs
# guest_df.to_csv(f"{output_dir}guest.csv", index=False)
# for k in ['calendar', 'weather', 'school_holidays', 'public_holidays']:
#     filtered[k].to_csv(f"{output_dir}{k}.csv", index=False)

# ----------------------------------------
# Build and save merged ordinal-rank dataset
# ----------------------------------------
full_rank_int_df = guest_df.copy()
for k in ['calendar', 'weather', 'school_holidays', 'public_holidays']:
    full_rank_int_df = full_rank_int_df.merge(filtered[k], on="Date", how="left")
full_rank_int_df = full_rank_int_df.merge(filtered['rank_int'], on="Date", how="left")

full_rank_int_df.to_csv(f"{output_dir}full_restaurant_data_rank_int.csv", index=False)
print(f"Saved: full_restaurant_data_rank_int.csv — {full_rank_int_df.shape}")

# ----------------------------------------
# Build and save merged nested one-hot dataset
# ----------------------------------------
full_rank_1hot_df = guest_df.copy()
for k in ['calendar', 'weather', 'school_holidays', 'public_holidays']:
    full_rank_1hot_df = full_rank_1hot_df.merge(filtered[k], on="Date", how="left")
full_rank_1hot_df = full_rank_1hot_df.merge(filtered['rank_1hot'], on="Date", how="left")

full_rank_1hot_df.to_pickle(f"{output_dir}full_restaurant_data_rank_1hot.pkl")
print(f"Saved: full_restaurant_data_rank_1hot.pkl — {full_rank_1hot_df.shape}")


# 4. Sanity-check: they should all have the same shape and date range
for name, df in filtered.items():
    print(f"{name:15s} -> {df.shape[0]} rows, {df['Date'].min().date()}–{df['Date'].max().date()}")

Saved: full_restaurant_data_rank_int.csv — (1777, 58)
Saved: full_restaurant_data_rank_1hot.pkl — (1777, 58)
calendar        -> 1777 rows, 2019-01-01–2025-04-28
weather         -> 1777 rows, 2019-01-01–2025-04-28
school_holidays -> 1777 rows, 2019-01-01–2025-04-28
public_holidays -> 1777 rows, 2019-01-01–2025-04-28
rank_int        -> 1777 rows, 2019-01-01–2025-04-28
rank_1hot       -> 1777 rows, 2019-01-01–2025-04-28


In [33]:
# -----------------------------------------------------------
# Build and save restaurant data with number of sold articles
# -----------------------------------------------------------

full_restaurant_sold_articles_int_df = guest_df.copy()
number_sold_items_df = course_data_df.copy()

# Filterint sold items dataframe by date
number_sold_items_df = number_sold_items_df[
    number_sold_items_df['Date'].isin(valid_dates) &
    (~in_covid_period(number_sold_items_df["Date"]))
].reset_index(drop=True)

# Build the dataframe with number of sold articles included
for k in ['calendar', 'weather', 'school_holidays', 'public_holidays']:
    full_restaurant_sold_articles_int_df = full_restaurant_sold_articles_int_df.merge(filtered[k], on="Date", how="left")
full_restaurant_sold_articles_int_df = full_restaurant_sold_articles_int_df.merge(number_sold_items_df, on="Date", how="left")

# Saving the dataframe with sold articles included
full_restaurant_sold_articles_int_df.to_csv(f"{output_dir}full_restaurant_data_sold_articles_int.csv", index=False)
print(f"Saved: full_restaurant_sold_articles_int.csv - {full_restaurant_sold_articles_int_df.shape}")

full_restaurant_sold_articles_int_df.head()


Saved: full_restaurant_sold_articles_int.csv - (1777, 58)


,Date,GUESTS,is_Friday,is_Monday,is_Saturday,is_Sunday,is_Thursday,is_Tuesday,is_Wednesday,tempmax,...,art_poffert,art_sate_spies,art_schnitzel,art_sliptong,art_sorbet,art_spareribs,art_stamppot,art_tournedos,art_vers_van_de_markt,art_zalmfilet
0,2019-01-01,89.0,0,0,0,0,0,1,0,8.8,...,4,7,6,5,0,6,0,0,0,3
1,2019-01-02,111.0,0,0,0,0,0,0,1,7.0,...,3,9,5,5,0,6,0,0,0,2
2,2019-01-03,114.0,0,0,0,0,1,0,0,6.1,...,0,8,5,9,0,17,0,0,0,4
3,2019-01-04,93.0,1,0,0,0,0,0,0,7.0,...,1,7,3,4,1,2,0,0,0,5
4,2019-01-05,143.0,0,0,1,0,0,0,0,8.0,...,7,12,3,10,2,2,0,0,0,11


In [1]:
import pandas as pd
import numpy as np
import os
import pickle as pkl

np.set_printoptions(threshold=np.inf)

# Loading guest data from the restaurant
guest_df = pd.read_csv("../data/raw/WEEVA_GUESTS.csv")
guest_df.rename(columns={"DATE" : 'Date'}, inplace=True)
guest_df["Date"] = pd.to_datetime(guest_df["Date"])

# Loading weather data by time frame
weather_nov_2018_may_2021_df       = pd.read_csv("../data/raw/Groningen 2018-11-01 to 2021-05-31.csv")
weather_june_2021_december_2023_df = pd.read_csv("../data/raw/Groningen 2021-06-01 to 2023-12-31.csv")
weather_jan_2024_april_2025_df     = pd.read_csv("../data/raw/Groningen 2024-01-01 to 2025-04-28.csv")
weather_df = pd.concat([weather_nov_2018_may_2021_df, 
                        weather_june_2021_december_2023_df, 
                        weather_jan_2024_april_2025_df], 
                        ignore_index=True)
weather_df.rename(columns={"datetime" : 'Date'}, inplace=True)
weather_df["Date"] = pd.to_datetime(weather_df["Date"])
irrelevant_features = ['name', 'dew', 'precipcover', 'snow', 'snowdepth',
                        'winddir', 'sealevelpressure', 'visibility', 
                        'solarenergy', 'severerisk', 'sunrise', 'sunset',
                        'moonphase', 'description', 'stations', 'icon', 
                        'conditions']
weather_df = weather_df.drop(columns=irrelevant_features)
weather_df['rain'] = weather_df['preciptype'].str.contains('rain', na=False).astype(int)
weather_df['snow'] = weather_df['preciptype'].str.contains('snow', na=False).astype(int)
weather_df = weather_df.drop(columns='preciptype')

# Loading data on holidays and calendar dates
school_holidays_df           = pd.read_csv("../data/raw/groningen_school_holidays_boolean.csv")
public_holidays_groningen_df = pd.read_csv("../data/raw/public_holidays_2018_2025.csv")
public_holidays_germany_df   = pd.read_csv("../data/raw/public_holidays_germany_2018_2025.csv")
calendar_df                  = pd.read_csv("../data/raw/dates_with_weekdays.csv")

# Calendar preprocessing
calendar_df["Date"] = pd.to_datetime(calendar_df["Date"])
dow_dummies = pd.get_dummies(calendar_df['DayOfWeek'], prefix='is', dtype=int)
calendar_df = pd.concat([calendar_df, dow_dummies], axis=1)
calendar_df = calendar_df.drop(columns=['DayOfWeek', 'IsWeekend']) 

# School holidays preprocessing
school_holidays_df["Date"] = pd.to_datetime(school_holidays_df["Date"])
start_date = '2018-11-01'
end_date = '2025-04-28'
school_holidays_df = school_holidays_df[
    (school_holidays_df['Date'] >= start_date) &
    (school_holidays_df['Date'] <= end_date)
]
school_holidays_df = school_holidays_df.drop_duplicates(subset='Date', keep='last')
school_holidays_bool_df = pd.DataFrame(school_holidays_df)
school_holidays_bool_df['IsHoliday'] = school_holidays_bool_df['IsHoliday'].map({"Yes": 1, "No": 0})

# Groningen public holidays preprocessing
public_holidays_groningen_df["Holiday"] = public_holidays_groningen_df['Holiday'].str.strip()
name_map = {
    "New Year": "New Year's Day",
    "Koningsdag (National Holiday)": "King\'s Day",
    "Koningsdag (National Day)": "King\'s Day",
    "St. Stephen's Day": "Second Christmas Day"
}
public_holidays_groningen_df["Holiday"] = public_holidays_groningen_df['Holiday'].replace(name_map)

# Germany public holidays preprocessing
public_holidays_germany_df["Holiday"] = public_holidays_germany_df['Holiday'].str.strip()
name_map = {
    "New Years Day": "New Year's Day",
    "Christmas Day": "Christmas",
    "Boxing Day": "Second Christmas Day"
}
public_holidays_germany_df["Holiday"] = public_holidays_germany_df['Holiday'].replace(name_map)

# Setting the date column to the right data type
public_holidays_groningen_df["Date"] = pd.to_datetime(public_holidays_groningen_df["Date"], format="%d.%m.%Y")
public_holidays_germany_df["Date"] = pd.to_datetime(public_holidays_germany_df["Date"], format="%d.%m.%Y")

# Combine all the holidays
combined_holidays_df = pd.concat([public_holidays_groningen_df, public_holidays_germany_df], ignore_index=True)
combined_holidays_df['is_holiday'] = 1
combined_holidays_df = combined_holidays_df.pivot_table(
    index='Date',
    columns='Holiday',
    values='is_holiday',
    fill_value=0
).reset_index()
date_range = pd.date_range(start='2018-11-01', end='2025-04-28', freq='D')
full_date_range_df = pd.DataFrame({'Date': date_range})
merged_df = full_date_range_df.merge(combined_holidays_df, on='Date', how='left')
final_holiday_df = merged_df.fillna('0').astype({col: 'int' for col in merged_df.columns if col != 'Date'})

# Loading data on number of items ordered in the restaurant
course_data_2018_2022_df = pd.read_csv("../data/raw/Weeva_data_2018-2022.csv")
course_data_2023_2025_df = pd.read_csv("../data/raw/Weeva_Data_2023-x.csv")
course_data_df = pd.concat([course_data_2018_2022_df, course_data_2023_2025_df], ignore_index=True)
course_data_df["Date"] = pd.to_datetime(course_data_df["Date"], format="%d-%m-%Y")
article_map = {
    "broodplankje": "art_broodplankje",
    "captain.*dinner": "art_captain_dinner",
    "(?<!\w\s)cr.me.*br.l.e(?!\s)": "art_creme_brulee",
    "dame.*blanche(?!\s)": "art_dame_blanche",
    "sliptong.*meuni.re": "art_sliptong",
    "garnalen.*cocktail": "art_garnalen_cocktail",
    "bloedworst": "art_bloedworst",
    "olijven": "art_olijven",
    "kaasplankje(?!\s)": "art_kaasplankje",
    "(?<!\w\s)kalfslever": "art_kalfslever",
    "koffie.*compleet(?!\s)": "art_koffie_compleet",
    "groningse poffert": "art_poffert",
    "runder.*carpaccio": "art_carpaccio",
    "sat.*spies(?!\s)": "art_sate_spies",
    "schnitzel": "art_schnitzel",
    "sorbet.*weeva": "art_sorbet",
    "andijvie stamppot": "art_stamppot",
    "vers.*markt": "art_vers_van_de_markt",
    "weeva.*gehaktbal(?!\s)": "art_gehaktbal",
    "weeva.*spareribs(?!\s)": "art_spareribs",
    "tournedos(?!\s)": "art_tournedos",
    "zalmfilet(?!\s)": "art_zalmfilet",
}
course_data_df["Article"] = course_data_df["Article"].str.lower()
course_data_df["Article"] = course_data_df["Article"].replace(article_map, regex=True)
valid_articles = set(article_map.values())
course_data_df = course_data_df[course_data_df["Article"].isin(valid_articles)]
course_data_df = course_data_df.pivot_table(
    index="Date",
    columns="Article",
    values="Sold articles amount",
    fill_value=0,
    aggfunc="sum",
).reset_index()

# Build rank tables
counts_only = course_data_df.set_index('Date')
article_names = counts_only.columns.tolist()
A = len(article_names)
rank_int_df = (
    counts_only
      .rank(axis=1, method='first', ascending=False)
      .astype(int)
      .reset_index()
)
def order_list(row):
    return list(row.sort_values(ascending=False).index)
rank_order_df = pd.DataFrame({
    'Date': counts_only.index,
    'rank_order': counts_only.apply(order_list, axis=1)
})
article_to_idx = {name: i for i, name in enumerate(article_names)}
rank_idx_vec_df = rank_order_df.copy()
rank_idx_vec_df['rank_idx_vec'] = rank_idx_vec_df['rank_order'].apply(lambda lst: [article_to_idx[a] for a in lst])
rank_idx_vec_df = rank_idx_vec_df.drop(columns='rank_order')
def to_onehot(rank, length=A):
    vec = [0] * length
    vec[rank - 1] = 1
    return vec
rank_1hot_nested_df = rank_int_df.copy()
for col in article_names:
    rank_1hot_nested_df[col] = rank_1hot_nested_df[col].apply(to_onehot)

# COVID MASKING
output_dir = "../data/cleaned/"
os.makedirs(output_dir, exist_ok=True)
COVID_WINDOWS = [
    ("2020-03-01", "2020-05-31"),
    ("2020-12-01", "2021-06-30"),
    ("2021-11-01", "2022-01-31"),
]
def in_covid_period(s, windows=COVID_WINDOWS):
    mask = pd.Series(False, index=s.index)
    for start, end in windows:
        mask |= s.between(start, end)
    return mask

# Align by shared dates
base_dfs = [
    guest_df,
    calendar_df,
    weather_df,
    school_holidays_bool_df,
    final_holiday_df,
    rank_int_df,
    rank_1hot_nested_df,
]
common_idx = pd.DatetimeIndex(base_dfs[0]["Date"].unique())
for df in base_dfs[1:]:
    common_idx = common_idx.intersection(pd.DatetimeIndex(df["Date"].unique()))
for i in range(len(base_dfs)):
    base_dfs[i] = base_dfs[i][base_dfs[i]["Date"].isin(common_idx)].reset_index(drop=True)
(guest_df,
 calendar_df,
 weather_df,
 school_holidays_bool_df,
 final_holiday_df,
 rank_int_df,
 rank_1hot_nested_df) = base_dfs

# Remove guest outliers + COVID dates
guest_df['GUESTS'] = pd.to_numeric(guest_df['GUESTS'], errors='coerce')
guest_df = guest_df[
    (guest_df['GUESTS'].between(1, 200)) &
    (~in_covid_period(guest_df['Date']))
].reset_index(drop=True)
valid_dates = set(guest_df['Date'])
filtered = {}
for name, df in {
    'calendar': calendar_df,
    'weather': weather_df,
    'school_holidays': school_holidays_bool_df,
    'public_holidays': final_holiday_df,
    'rank_int': rank_int_df,
    'rank_1hot': rank_1hot_nested_df,
}.items():
    df = df[
        df['Date'].isin(valid_dates) &
        (~in_covid_period(df["Date"]))
    ].reset_index(drop=True)
    filtered[name] = df

# Build and save merged ordinal-rank dataset
full_rank_int_df = guest_df.copy()
for k in ['calendar', 'weather', 'school_holidays', 'public_holidays']:
    full_rank_int_df = full_rank_int_df.merge(filtered[k], on="Date", how="left")
full_rank_int_df = full_rank_int_df.merge(filtered['rank_int'], on="Date", how="left")
full_rank_int_df.to_csv(f"{output_dir}full_restaurant_data_rank_int.csv", index=False)
print(f"Saved: full_restaurant_data_rank_int.csv — {full_rank_int_df.shape}")

# Build and save merged nested one-hot dataset
full_rank_1hot_df = guest_df.copy()
for k in ['calendar', 'weather', 'school_holidays', 'public_holidays']:
    full_rank_1hot_df = full_rank_1hot_df.merge(filtered[k], on="Date", how="left")
full_rank_1hot_df = full_rank_1hot_df.merge(filtered['rank_1hot'], on="Date", how="left")
full_rank_1hot_df.to_pickle(f"{output_dir}full_restaurant_data_rank_1hot.pkl")
print(f"Saved: full_restaurant_data_rank_1hot.pkl — {full_rank_1hot_df.shape}")

# Build and save restaurant data with number of sold articles
full_restaurant_sold_articles_int_df = guest_df.copy()
number_sold_items_df = course_data_df.copy()
number_sold_items_df = number_sold_items_df[
    number_sold_items_df['Date'].isin(valid_dates) &
    (~in_covid_period(number_sold_items_df["Date"]))
].reset_index(drop=True)
for k in ['calendar', 'weather', 'school_holidays', 'public_holidays']:
    full_restaurant_sold_articles_int_df = full_restaurant_sold_articles_int_df.merge(filtered[k], on="Date", how="left")
full_restaurant_sold_articles_int_df = full_restaurant_sold_articles_int_df.merge(number_sold_items_df, on="Date", how="left")
full_restaurant_sold_articles_int_df.to_csv(f"{output_dir}full_restaurant_data_sold_articles_int.csv", index=False)
print(f"Saved: full_restaurant_sold_articles_int.csv - {full_restaurant_sold_articles_int_df.shape}")

C:\Users\Bianca\AppData\Local\Temp\ipykernel_4284\4126890123.py:46: UserWarning: Parsing dates in %d.%m.%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  school_holidays_df["Date"] = pd.to_datetime(school_holidays_df["Date"])


Saved: full_restaurant_data_rank_int.csv — (1730, 58)
Saved: full_restaurant_data_rank_1hot.pkl — (1730, 58)
Saved: full_restaurant_sold_articles_int.csv - (1730, 58)
